# Project 5.2 — Lending Club Loan Data Analysis (Deep Learning)
**Predict whether a loan will default** (`not.fully.paid`) — binary classification on the
cleaned Lending Club dataset (9,578 rows x 14 cols), **imbalanced (~16% minority)**.

Follows `plan.md` and `explanation.md` in the same folder. Run top-to-bottom.

## 1. Project Overview & Business Problem
Flag loans likely to default. Defaults are the minority class, so we optimise **AUC / recall**, not accuracy.

## 2. Importing Required Libraries

In [ ]:
# Colab already has these, but this guarantees versions are present.
# (imbalanced-learn is only needed if you try the optional SMOTE path.)
!pip -q install imbalanced-learn >/dev/null 2>&1
print("setup cell done")

In [ ]:
import os, numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, roc_curve, precision_recall_curve,
                             average_precision_score, f1_score)
from sklearn.utils.class_weight import compute_class_weight

# Reproducibility
SEED = 42
np.random.seed(SEED); tf.random.set_seed(SEED)
pd.set_option('display.max_columns', 200)
sns.set(style='whitegrid')
print('TensorFlow', tf.__version__)

## 3. Loading the Dataset
Upload `loan_data (2).csv` (or point `path` to it in Drive).

In [ ]:
# ---- Flexible data loader (works in Colab or locally) ----
FILENAME = 'loan_data (2).csv'   # cleaned Lending Club dataset (9,578 x 14)

CANDIDATES = [
    FILENAME,
    '/content/' + FILENAME,
    'CEP_2_Datasets/' + FILENAME,
    '/content/drive/MyDrive/' + FILENAME,
]
path = next((p for p in CANDIDATES if os.path.exists(p)), None)

if path is None:
    try:
        from google.colab import files
        print('Please upload', FILENAME)
        uploaded = files.upload()
        path = list(uploaded.keys())[0]
    except Exception:
        raise FileNotFoundError('Upload the CSV or set path manually.')

df = pd.read_csv(path)
print('Loaded:', path, '| Shape:', df.shape)
df.head()

## 4-5. Data Understanding & Quality Checks
The cleaned file has no missing values — we still verify.

In [ ]:
print('Shape:', df.shape)
print('\nColumns:', list(df.columns))
print('\nTarget distribution (not.fully.paid):')
print(df['not.fully.paid'].value_counts())
print(df['not.fully.paid'].value_counts(normalize=True).round(4))
print('\nMissing values total:', int(df.isnull().sum().sum()))
df.describe()

## 6-8. Exploratory Data Analysis

In [ ]:
# Target balance
plt.figure(figsize=(5,4))
sns.countplot(x='not.fully.paid', data=df)
plt.title('Class balance (1 = default ~16%)'); plt.show()

# Numeric distributions
num_cols = ['int.rate','fico','dti','log.annual.inc','revol.util','installment']
df[num_cols].hist(figsize=(12,7), bins=30); plt.tight_layout(); plt.show()

In [ ]:
# purpose vs default rate
rate = df.groupby('purpose')['not.fully.paid'].mean().sort_values()
plt.figure(figsize=(7,4)); rate.plot(kind='barh')
plt.title('Default rate by purpose'); plt.xlabel('mean not.fully.paid'); plt.show()

# fico vs int.rate coloured by default
plt.figure(figsize=(6,4))
sns.scatterplot(x='fico', y='int.rate', hue='not.fully.paid', data=df, alpha=0.3)
plt.title('FICO vs interest rate'); plt.show()

In [ ]:
plt.figure(figsize=(9,7))
sns.heatmap(df.select_dtypes('number').corr(), annot=False, cmap='coolwarm', center=0)
plt.title('Correlation heatmap (numeric)'); plt.show()

## 9. Feature Transformation (Task 1) — one-hot encode `purpose`
`drop_first=True` avoids the dummy-variable trap.

In [ ]:
# --- Feature Transformation (Task 1): one-hot encode 'purpose' ---
df_enc = pd.get_dummies(df, columns=['purpose'], drop_first=True)
df_enc = df_enc.astype(np.float32)
print('Shape after encoding:', df_enc.shape)
df_enc.head()

## 10. Correlation Analysis & Feature Selection (Task 3)

In [ ]:
# --- Correlation-based feature selection (Task 3) ---
feat = df_enc.drop(columns=['not.fully.paid'])
corr = feat.corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
to_drop = [c for c in upper.columns if (upper[c] > 0.90).any()]
print('Highly-correlated columns dropped:', to_drop)
feat = feat.drop(columns=to_drop)
print('Final feature count:', feat.shape[1])

## 11-12. Feature/Target Separation & Train/Val/Test Split

In [ ]:
# --- Feature/Target separation + 3-way stratified split ---
X = feat.values
y = df_enc['not.fully.paid'].astype(int).values

X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=SEED)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.20, stratify=y_trainval, random_state=SEED)
print('train:', X_train.shape, '| val:', X_val.shape, '| test:', X_test.shape)

## 13-14. Scaling & Class Imbalance Handling
Scaler fit on train only; balanced class weights for the ~16% minority.

In [ ]:
# --- Scale: FIT ON TRAIN ONLY ---
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)

# --- Class weights (train only) ---
classes = np.unique(y_train)
weights = compute_class_weight('balanced', classes=classes, y=y_train)
class_weight = dict(zip(classes, weights))
print('class_weight =', class_weight)

## 15. Deep Learning Model Architecture

In [ ]:
# --- Keras Sequential model (sized small for a small dataset) ---
n_features = X_train.shape[1]
model = keras.Sequential([
    layers.Input(shape=(n_features,)),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(16, activation='relu'),
    layers.Dense(1, activation='sigmoid'),
])
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy', keras.metrics.AUC(name='auc'),
             keras.metrics.Precision(name='precision'),
             keras.metrics.Recall(name='recall')],
)
model.summary()

## 16-17. Training & Overfitting Analysis

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(monitor='val_auc', mode='max',
                                  patience=10, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4),
]
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=150, batch_size=64,
    class_weight=class_weight,
    callbacks=callbacks, verbose=1)

In [ ]:
h = history.history
fig, ax = plt.subplots(1, 2, figsize=(12,4))
ax[0].plot(h['loss'], label='train'); ax[0].plot(h['val_loss'], label='val')
ax[0].set_title('Loss'); ax[0].legend()
ax[1].plot(h['auc'], label='train'); ax[1].plot(h['val_auc'], label='val')
ax[1].set_title('AUC'); ax[1].legend(); plt.show()

## 18-19. Evaluation & Threshold Selection (validation)

In [ ]:
val_proba = model.predict(X_val).ravel()
print('Validation ROC-AUC: %.4f' % roc_auc_score(y_val, val_proba))
prec, rec, thr = precision_recall_curve(y_val, val_proba)
f1s = 2*prec*rec/(prec+rec+1e-9)
best_idx = int(np.nanargmax(f1s))
best_threshold = thr[best_idx] if best_idx < len(thr) else 0.5
print('Best threshold (max F1 on val): %.3f  F1=%.3f' % (best_threshold, f1s[best_idx]))
val_pred = (val_proba >= best_threshold).astype(int)
print(classification_report(y_val, val_pred, digits=3))

## 20. Final Test Evaluation

In [ ]:
test_proba = model.predict(X_test).ravel()
test_pred  = (test_proba >= best_threshold).astype(int)
print('TEST ROC-AUC : %.4f' % roc_auc_score(y_test, test_proba))
print('TEST PR-AUC  : %.4f' % average_precision_score(y_test, test_proba))
print('TEST Accuracy: %.4f' % (test_pred == y_test).mean())
print('\nClassification report (TEST):')
print(classification_report(y_test, test_pred, digits=3))
cm = confusion_matrix(y_test, test_pred)
plt.figure(figsize=(4.5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted'); plt.ylabel('Actual'); plt.title('Confusion Matrix (TEST)'); plt.show()

## 21. Save the Model

In [ ]:
model.save('lending_club_model.keras')
print('Saved lending_club_model.keras')
# from google.colab import files; files.download('lending_club_model.keras')

## 22. Business Insights, Limitations & Conclusion
- **Read AUC / recall**, not accuracy (predicting "all paid" already scores ~84%).
- Strongest default signals: `int.rate`, `fico`, `inq.last.6mths`, `credit.policy`.
- **Limitations:** small dataset; a neural net may not beat gradient-boosted trees here.
  **Next steps:** try SMOTE, tune layers/dropout, or benchmark LightGBM/XGBoost.